# E-Commerce Sales Prediction Using Big Data Analytics

**Dataset:** UCI Online Retail II  
**Stack:** HDFS → Spark/PySpark → Spark ML → Visualization  
**Objective:** Analyze sales patterns and predict transaction-day revenue.

### Project Flow
Load cleaned data from HDFS → perform Spark SQL analytics → aggregate daily sales → engineer time-series features → chronological train/test split → train and compare regression models → visualize results.


## 1. Initialize Spark and Load Libraries


In [1]:
# Initialize the Spark environment and import the libraries required for distributed analytics and ML.
# Keeping all project dependencies together makes the notebook flow easier to reproduce.

from pyspark.sql import SparkSession
from pyspark.sql.functions import col, to_date, sum as spark_sum, lag, avg, dayofweek, month, year
from pyspark.sql.window import Window
from pyspark.ml.feature import VectorAssembler
from pyspark.ml.regression import LinearRegression, RandomForestRegressor
from pyspark.ml.evaluation import RegressionEvaluator
import csv
import matplotlib.pyplot as plt

spark = SparkSession.builder.appName("EcommerceSalesPrediction").getOrCreate()
spark.sparkContext.setLogLevel("WARN")

## 2. Load Cleaned Dataset from HDFS


In [ ]:
# Read the cleaned e-commerce dataset directly from HDFS using Spark's distributed CSV reader.
# Infer the schema so numeric sales fields can be used directly in downstream analytics.

hdfs_path = "hdfs://namenode:8020/data/ecommerce/raw/cleaned_online_retail.csv"

df = (
    spark.read
    .option("header", True)
    .option("inferSchema", True)
    .csv(hdfs_path)
)

print("Records:", df.count())
df.printSchema()

## 3. Perform Initial Sales Analytics


In [ ]:
# Create a temporary Spark SQL view to analyze revenue by country and month.
# These queries identify important sales patterns before building the prediction model.

df.createOrReplaceTempView("ecommerce_sales")

country_revenue = spark.sql("""
    SELECT Country, SUM(Revenue) AS Total_Revenue
    FROM ecommerce_sales
    GROUP BY Country
    ORDER BY Total_Revenue DESC
    LIMIT 10
""")
country_revenue.show(10, truncate=False)

monthly_sales = spark.sql("""
    SELECT Year, Month,
           SUM(Quantity) AS Total_Quantity,
           SUM(Revenue) AS Total_Revenue
    FROM ecommerce_sales
    GROUP BY Year, Month
    ORDER BY Year, Month
""")
monthly_sales.show(30, truncate=False)

## 4. Aggregate Sales by Transaction Day


In [ ]:
# Aggregate transaction records into daily sales values to create the prediction target.
# Using transaction days keeps the time series aligned with days on which sales were recorded.

daily_sales = spark.sql("""
    SELECT
        to_date(InvoiceDate) AS SalesDate,
        SUM(Quantity) AS Daily_Quantity,
        SUM(Revenue) AS Daily_Revenue
    FROM ecommerce_sales
    GROUP BY to_date(InvoiceDate)
    ORDER BY SalesDate
""")

daily_sales.show(10, truncate=False)
print("Transaction days:", daily_sales.count())

## 5. Create Time-Series Prediction Features


In [ ]:
# Create lag and rolling-average features so previous sales patterns can help predict future revenue.
# Add calendar features to capture weekly and seasonal effects in the sales data.

window_order = Window.orderBy("SalesDate")

prediction_df = (
    daily_sales
    .withColumn("Lag_1_Day", lag("Daily_Revenue", 1).over(window_order))
    .withColumn("Lag_7_Days", lag("Daily_Revenue", 7).over(window_order))
    .withColumn("Lag_14_Days", lag("Daily_Revenue", 14).over(window_order))
)

rolling_window = Window.orderBy("SalesDate").rowsBetween(-7, -1)

prediction_df = prediction_df.withColumn(
    "Rolling_7_Day_Avg",
    avg("Daily_Revenue").over(rolling_window)
)

prediction_df = prediction_df.withColumn("DayOfWeek_Num", dayofweek("SalesDate"))
prediction_df = prediction_df.withColumn("Month_Num", month("SalesDate"))
prediction_df = prediction_df.withColumn("Year_Num", year("SalesDate"))

prediction_df = prediction_df.dropna()

print("Prediction records:", prediction_df.count())
prediction_df.show(5, truncate=False)

## 6. Split Data Chronologically


In [ ]:
# Split the time-series data in chronological order so future observations are never used for training.
# The first 80% is used for training and the remaining 20% is reserved for unseen testing.

total_records = prediction_df.count()
train_size = int(total_records * 0.8)

prediction_df = prediction_df.orderBy("SalesDate")
train_df = prediction_df.limit(train_size)
test_df = prediction_df.subtract(train_df).orderBy("SalesDate")

print("Total:", total_records)
print("Train:", train_df.count())
print("Test:", test_df.count())

print("Train period:")
train_df.select("SalesDate").orderBy("SalesDate").show(1)

print("Test period:")
test_df.select("SalesDate").orderBy("SalesDate").show(1)

## 7. Train Baseline Linear Regression


In [ ]:
# Train a baseline Linear Regression model using only historical revenue-based features.
# Evaluate the baseline with RMSE, MAE, and R² to establish a reference for later models.

baseline_features = [
    "Lag_1_Day",
    "Lag_7_Days",
    "Lag_14_Days",
    "Rolling_7_Day_Avg"
]

assembler_baseline = VectorAssembler(
    inputCols=baseline_features,
    outputCol="features"
)

train_baseline = assembler_baseline.transform(train_df)
test_baseline = assembler_baseline.transform(test_df)

lr_baseline = LinearRegression(
    featuresCol="features",
    labelCol="Daily_Revenue"
)

lr_baseline_model = lr_baseline.fit(train_baseline)
baseline_predictions = lr_baseline_model.transform(test_baseline)

rmse_eval = RegressionEvaluator(
    labelCol="Daily_Revenue",
    predictionCol="prediction",
    metricName="rmse"
)
mae_eval = RegressionEvaluator(
    labelCol="Daily_Revenue",
    predictionCol="prediction",
    metricName="mae"
)
r2_eval = RegressionEvaluator(
    labelCol="Daily_Revenue",
    predictionCol="prediction",
    metricName="r2"
)

baseline_rmse = rmse_eval.evaluate(baseline_predictions)
baseline_mae = mae_eval.evaluate(baseline_predictions)
baseline_r2 = r2_eval.evaluate(baseline_predictions)

print("Baseline LR:", baseline_rmse, baseline_mae, baseline_r2)

## 8. Train Improved Linear Regression


In [ ]:
# Train an improved Linear Regression model by adding calendar-based features to the baseline inputs.
# Compare its performance with the baseline using the same unseen test period and metrics.

improved_features = baseline_features + [
    "DayOfWeek_Num",
    "Month_Num",
    "Year_Num"
]

assembler_improved = VectorAssembler(
    inputCols=improved_features,
    outputCol="features"
)

train_improved = assembler_improved.transform(train_df)
test_improved = assembler_improved.transform(test_df)

lr_improved = LinearRegression(
    featuresCol="features",
    labelCol="Daily_Revenue"
)

lr_improved_model = lr_improved.fit(train_improved)
improved_predictions = lr_improved_model.transform(test_improved)

improved_rmse = rmse_eval.evaluate(improved_predictions)
improved_mae = mae_eval.evaluate(improved_predictions)
improved_r2 = r2_eval.evaluate(improved_predictions)

print("Improved LR:", improved_rmse, improved_mae, improved_r2)

## 9. Train Random Forest Regression


In [ ]:
# Train Random Forest Regression to capture nonlinear relationships among the engineered features.
# Evaluate it on the same test data so its performance can be compared fairly with both Linear Regression models.

rf = RandomForestRegressor(
    featuresCol="features",
    labelCol="Daily_Revenue",
    numTrees=100,
    maxDepth=8,
    seed=42
)

rf_model = rf.fit(train_improved)
rf_predictions = rf_model.transform(test_improved)

rf_rmse = rmse_eval.evaluate(rf_predictions)
rf_mae = mae_eval.evaluate(rf_predictions)
rf_r2 = r2_eval.evaluate(rf_predictions)

print("Random Forest:", rf_rmse, rf_mae, rf_r2)

rf_predictions.select(
    "SalesDate", "Daily_Revenue", "prediction"
).orderBy("SalesDate").show(10, truncate=False)

## 10. Compare Model Performance


In [ ]:
# Store the evaluation metrics of all three models in one comparison table and CSV file.
# This provides a consistent basis for selecting the best-performing prediction model.

results = [
    ["Linear Regression - Baseline", baseline_rmse, baseline_mae, baseline_r2],
    ["Linear Regression - Improved", improved_rmse, improved_mae, improved_r2],
    ["Random Forest Regression", rf_rmse, rf_mae, rf_r2]
]

with open("/home/jovyan/work/model_comparison.csv", "w", newline="") as file:
    writer = csv.writer(file)
    writer.writerow(["Model", "RMSE", "MAE", "R2"])
    writer.writerows(results)

print("Model comparison saved.")

## Visualizations

## 11. Visualization: Monthly Revenue Trend


In [ ]:
# Visualize monthly revenue over time to identify seasonal changes and high-sales periods.
# Save the chart as a project result so it can be reused in the report and presentation.

rows = monthly_sales.collect()
months = [f"{row['Year']}-{int(row['Month']):02d}" for row in rows]
revenue = [float(row['Total_Revenue']) for row in rows]

plt.figure(figsize=(12, 5))
plt.plot(months, revenue, marker="o")
plt.title("Monthly Revenue Trend")
plt.xlabel("Month")
plt.ylabel("Total Revenue")
plt.xticks(rotation=45)
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig("/home/jovyan/work/monthly_revenue_trend.png", dpi=200, bbox_inches="tight")
plt.show()

## 12. Visualization: Top 10 Countries by Revenue


In [ ]:
# Visualize the ten countries with the highest total revenue in the cleaned dataset.
# A horizontal bar chart makes the large differences between countries easier to compare.

rows = country_revenue.collect()
countries = [row["Country"] for row in rows][::-1]
revenue = [float(row["Total_Revenue"]) for row in rows][::-1]

plt.figure(figsize=(10, 6))
plt.barh(countries, revenue)
plt.title("Top 10 Countries by Revenue")
plt.xlabel("Total Revenue")
plt.ylabel("Country")
plt.tight_layout()
plt.savefig("/home/jovyan/work/top_10_countries_revenue.png", dpi=200, bbox_inches="tight")
plt.show()

## 13. Visualization: Model RMSE Comparison


In [ ]:
# Compare the RMSE values of the three prediction models to show which model has the lowest error.
# Save the comparison chart as evidence for the final model-selection decision.

models = [row[0] for row in results]
rmse_values = [float(row[1]) for row in results]

plt.figure(figsize=(10, 5))
plt.bar(models, rmse_values)
plt.title("Model Comparison - RMSE")
plt.xlabel("Model")
plt.ylabel("RMSE")
plt.xticks(rotation=15)
plt.tight_layout()
plt.savefig("/home/jovyan/work/model_rmse_comparison.png", dpi=200, bbox_inches="tight")
plt.show()

## 14. Visualization: Actual vs Predicted Revenue


In [ ]:
# Compare actual revenue with Random Forest predictions across the unseen test period.
# This visualization shows how closely the selected model follows real transaction-day revenue.

prediction_rows = rf_predictions.select(
    "SalesDate", "Daily_Revenue", "prediction"
).orderBy("SalesDate").collect()

dates = [row["SalesDate"] for row in prediction_rows]
actual = [float(row["Daily_Revenue"]) for row in prediction_rows]
predicted = [float(row["prediction"]) for row in prediction_rows]

plt.figure(figsize=(14, 6))
plt.plot(dates, actual, label="Actual Revenue")
plt.plot(dates, predicted, label="Predicted Revenue")
plt.title("Actual vs Predicted Daily Revenue - Random Forest")
plt.xlabel("Sales Date")
plt.ylabel("Revenue")
plt.legend()
plt.xticks(rotation=45)
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig("/home/jovyan/work/actual_vs_predicted_revenue.png", dpi=200, bbox_inches="tight")
plt.show()

## Conclusion

Random Forest Regression achieved the lowest RMSE and MAE and the highest R² among the three evaluated models. The visualizations also highlight monthly revenue trends, major revenue-generating countries, model error comparison, and the relationship between actual and predicted transaction-day revenue.
